In [6]:
import ipaddress
from pydantic import BaseModel, Field, field_validator, ValidationError

class Server(BaseModel):
    # Add variables in python way 
    # Field() helps you to add extra rules, description, constraints
    hostname: str = Field(min_length=5, max_length=20)
    ip_addr: str
    environment: str = Field(default="dev")

    #Replacing @setter with field_validator, this will automatically intercepts 'ip_addr' during initialization
    @field_validator('ip_addr')

    # Using decorator to define a class method
    @classmethod
    #'cls' will automatically point to 'self' as we are using the classmethod decorator
    # '-> str' specifies that function will return a string object back
    def validate_ip_format(cls, value: str) -> str:
        clean_ip = value.strip()
        try:
            # Checking if the given ip address is in correct format or not
            ipaddress.ip_address(clean_ip)
        except ValueError:
            return ValueError(f"'{value}' is not a valid IP address format.")
        except ValidationError as ve:
            return ve.json(indent=2)
        
        return clean_ip #This validator must return a cleaned_ip

In [7]:
# Declaring 'Server' Object with correct hostname length and ip format
myserver = Server(hostname="webserver-1", ip_addr="10.10.10.1", environment='prod')

In [8]:
# Printing values
print(f"Hostname : {myserver.hostname}")
print(f"IP Address : {myserver.ip_addr}")
print(f"Environment : {myserver.environment}")


Hostname : webserver-1
IP Address : 10.10.10.1
Environment : prod


In [10]:
# Declaring a Server object with wrong length and ip format
try:
    # Intentionally sending a broken hostname (too short) and a bad IP format
    broken_server = Server(hostname="web", ip_addr="10.10.10.2555")
except ValidationError as ve:
    print("🔒 Pydantic successfully intercepted the errors:\n")
    print(ve.json(indent=2))

🔒 Pydantic successfully intercepted the errors:

[
  {
    "type": "string_too_short",
    "loc": [
      "hostname"
    ],
    "msg": "String should have at least 5 characters",
    "input": "web",
    "ctx": {
      "min_length": 5
    },
    "url": "https://errors.pydantic.dev/2.13/v/string_too_short"
  }
]


In [ ]:
# Exercise 2:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int
    email: str
    is_active: bool = True

# Creating the User object with correct parameters
try:
    user1 = User(name="Alice", age=30)
    print(user1)
except Exception as e:
    print(str(e))

name='Alice' age=30 is_active=True


In [16]:
# Creating an user with improper parameters
try:
    user1 = User(name="Alice", age="thirty")
    print(user1)
except Exception as e:
    print(str(e))

1 validation error for User
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='thirty', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


In [17]:
# EXporting models to Dictionary
user1_dict = user1.model_dump()
print(user1_dict)

{'name': 'Alice', 'age': 30, 'is_active': True}


# Testing field_validator hook

In [ ]:
# @field_validator Example
from pydantic import BaseModel, field_validator
class User(BaseModel):
    user_name: str

    @field_validator("user_name") #validation hook between the 'user_name' and validate_name()
    @classmethod
    def validate_name(cls, v):
        if not v:
            raise ValueError("Name Can not be empty")
        return v

In [20]:

# Declaring an object with proper user name 
try:
    user1 = User(user_name="Souvik Dhar")
    print(user1)
except Exception as e:
    print(f"Error => {str(e)}")

user_name='Souvik Dhar'


In [21]:
# Declaring an invalid user
try:
    user2 = User(user_name="")
    print(user2)
except Exception as e:
    print(f"Error => {str(e)}")

Error => 1 validation error for User
user_name
  Value error, Name Can not be empty [type=value_error, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


# Example 2 for field_validator


In [1]:
from pydantic import BaseModel, Field, field_validator, EmailStr
class UserProfile(BaseModel):
    username: str = Field (
        description = "Th unique handle of the user",
        example = ["johndoe_99"]
    )
    password: str = Field(
        min_length = 8,
        max_length = 15,
        pattern=r'^(?=.*[0-9])(?=.*[^A-Za-z0-9]).+$',
        description="Password must contain at least one number and one special character."
    )
    user_age: int = Field(
        default = 18,
        ge = 18,
        le = 101
    )
    email: EmailStr = Field(
        alias="emailAddress"
    )

/tmp/ipykernel_10485/241178470.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  username: str = Field (


ImportError: email-validator is not installed, run `pip install 'pydantic[email]'`

In [ ]:
# Creating a good user profile
try:
    user1_profile = UserProfile(username="", password="", user_age=20, email="user1@xyz.com")
except Exception as e:
    print(f"Error occured \n {str(e)}")

if user1_profile:
    print(f"User Name : {user1_profile.username}")
    print(f"User Age : {user1_profile.age}")
    print(f"User Email : {user1_profile.email}")

#Note: Now pass blank user name or improper password to check how this behaves

# Example 3 - Implementing Class Inheritence and Pydantic 

Declaring Parent Class

In [ ]:
from pydantic import BaseModel, Field, field_validator
import ipaddress
from typing import Literal

Allowed_Environments = Literal["sbx", "dev", "tst", "npr", "prd"]
class Server(BaseModel):
    hostname: str = Field(
        description="Name of the server. Must be exactly 10 characters long. Example [ srvus1w001| srvus2d001]",
        min_length=10,
        max_length=10
        
    )
    ip_address: str
    environment: Allowed_Environments = Field(
        description="Value within [Sbx (Sandbox) | Dev (Development) | Tst (Test) | Npr (Non-Prod) | Prd (Prod) | ]",
        min_length=3,
        max_length=3
    )

    # Creating a hook for field_validator
    @field_validator('ip_address')
    # Creating an abstract function which will not be used by the child class directly
    @classmethod
    def validate_ip_address(cls, ip_addr) -> str:
        clean_ip = ip_addr.strip()
        try:
            ipaddress.ip_address(clean_ip)
        except Exception as e:
            raise ValueError(f"'{ip_addr}' is not in proper format")
        
        return clean_ip


Declaring a child class which will inherit from Server Class

In [42]:
class LinServer(Server):
    # This child class inherits hostname, ip_address and environment
    kernel: str
    is_patched: bool = False

    def get_values(self) -> str:
        response='\n'.join([
            f"Hostname : {self.hostname}", 
            f"IP : {self.ip_address}", 
            f"Environment: {self.environment}", 
            f"Kernel Ver : {self.kernel}", 
            f"Patch Status : {str(self.is_patched)}"
            ]
        )

        return response

Creating a Linux Class Object and use that

In [43]:
lbox1 = LinServer(
    hostname="srvus1w001",
    ip_address="10.10.1.10",
    environment="prd",
    kernel="6.1.0-amd64",
    is_patched=True
)

In [44]:
#Extracting the value
print(lbox1.get_values())

Hostname : srvus1w001
IP : 10.10.1.10
Environment: prd
Kernel Ver : 6.1.0-amd64
Patch Status : True


# Checking Model Validator Example

Imagine we are writing a script to enforce cloud infrastructure compliance rules:
- if the server environment is prod it must have is_patched set to true
- if the server environment is not production then it should not production ip address range (ex: 10.200.)

In [51]:
from pydantic import BaseModel, Field, field_validator, model_validator, ValidationError
from typing import Literal
import ipaddress

Allowed_Environments = Literal["sbx", "dev", "tst", "npr", "prd"]

class ServerProfile(BaseModel):
    hostname: str = Field(min_length=10, max_length=12)
    ip_addr: str
    environment: Allowed_Environments
    is_patched: bool = False

    @field_validator('ip_addr')
    @classmethod
    def validate_ip(cls, value) -> str:
        clean_ip = value.strip()
        try:
            ipaddress.ip_address(clean_ip)
        except Exception as e:
            raise (f"'{value}' is not in proper format")
        
        return clean_ip
    
    @model_validator(mode='after')
    def enforce_compliance_policies(self) -> 'ServerProfile':
        #Rule-1: Cross examine environment and is patched

        if self.environment == 'prd' and not self.is_patched:
            raise ValueError(
                f"Compliance Failure : Server '{self.hostname}' is designed as production"
                f"Currently is UNPATCHED. Deployment Blocked"
            )
        
        # Rule-2: Cross Examining environment and IP Address Range
        if self.environment != 'prd' and self.ip_addr.startswith("10.200."):
            raise ValueError(
                f"Security Routing Failure: '{self.environment}' Server '{self.hostname}' cannot use"
                f"Production IP Address Range ({self.ip_addr})"
            )
        
        return self

    def _get(self) -> str:
            result = '\n'.join([
                f"Server Hostname : {self.hostname}",
                f"IP : {self.ip_addr}",
                f"Env : {self.environment}",
                f"Patched : {str(self.is_patched)}"
            ])

            return result

Testing with valid parameters

In [ ]:
try:
    valid_server = ServerProfile(
        hostname="kus1wsrv001",
        ip_addr="10.200.1.10",
        environment="prd",
        is_patched=True
    )
    print("✅ Success! Compliant server verified and logged into database.")
except ValidationError as e:
    print(e.json(indent=2))

✅ Success! Compliant server verified and logged into database.


In [53]:
# Extracting the values for a valid server
print(valid_server._get())

Server Hostname : kus1wsrv001
IP : 10.200.1.10
Env : prd
Patched : True


Testing with Invalid Parameters

In [ ]:
try:
    bad_server = ServerProfile(
        hostname="kus1wsrv001",
        ip_addr="10.200.1.10",
        environment="npr",
        is_patched=True
    )
    print("✅ Success! Compliant server verified and logged into database.")
except ValidationError as e:
    print(f"🔒 Pydantic Compliance Engine Blocked Instantiation")
    print(e.json(indent=2))

[
  {
    "type": "value_error",
    "loc": [],
    "msg": "Value error, Security Routing Failure: 'npr' Server 'kus1wsrv001' cannot useProduction IP Address Range (10.200.1.10)",
    "input": {
      "hostname": "kus1wsrv001",
      "ip_addr": "10.200.1.10",
      "environment": "npr",
      "is_patched": true
    },
    "ctx": {
      "error": "Security Routing Failure: 'npr' Server 'kus1wsrv001' cannot useProduction IP Address Range (10.200.1.10)"
    },
    "url": "https://errors.pydantic.dev/2.13/v/value_error"
  }
]
